In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
from torch.utils.data import DataLoader, TensorDataset, random_split
from torchvision.models import resnet34,resnet18,resnet50

In [2]:
def fgsm_attack(model, images, labels, criterion, epsilon):
    images = images.clone().detach().requires_grad_(True)
    outputs = model(images)
    loss = criterion(outputs, labels)
    model.zero_grad()
    loss.backward()
    
    data_grad = images.grad.data
    perturbed_image = images + epsilon * data_grad.sign()
    perturbed_image = torch.clamp(perturbed_image, 0, 1)
    return perturbed_image.detach()


def pgd_attack(model, images, labels, criterion, epsilon, alpha, iters):
    perturbed_images = images.clone().detach()
    perturbed_images = perturbed_images + torch.empty_like(perturbed_images).uniform_(-epsilon, epsilon)
    perturbed_images = torch.clamp(perturbed_images, 0, 1)
    
    for _ in range(iters):
        perturbed_images.requires_grad = True
        outputs = model(perturbed_images)
        loss = criterion(outputs, labels)
        model.zero_grad()
        loss.backward()
        
        with torch.no_grad():
            perturbed_images = perturbed_images + alpha * perturbed_images.grad.sign()
            eta = torch.clamp(perturbed_images - images, min=-epsilon, max=epsilon)
            perturbed_images = torch.clamp(images + eta, 0, 1)
    return perturbed_images.detach()

In [3]:
data = np.load("train.npz") 
images = torch.from_numpy(data["images"]).float() / 255.0
labels = torch.from_numpy(data["labels"]).long()

full_dataset = TensorDataset(images, labels)


VAL_SIZE = 1000
train_size = len(full_dataset) - VAL_SIZE
train_dataset, val_dataset = random_split(
    full_dataset, [train_size, VAL_SIZE], 
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)

print(f"Train size: {len(train_dataset)} | Val size: {len(val_dataset)}")
print("Image shape:", images.shape)

Train size: 49000 | Val size: 1000
Image shape: torch.Size([50000, 3, 32, 32])


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

NUM_CLASSES = 9
model = resnet50(weights=None)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model = model.to(device)

model.eval()
with torch.no_grad():
    dummy_input = torch.randn(1, 3, 32, 32).to(device)
    out = model(dummy_input)
print("Output shape verification:", out.shape)

Training on device: cuda
Output shape verification: torch.Size([1, 9])


In [5]:
@torch.no_grad()
def evaluate_clean(model, loader):
    model.eval()
    correct, total = 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        pred = model(x).argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)
    return correct / total

def evaluate_pgd(model, loader, eps, alpha, iters):
    model.eval()
    correct, total = 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        x_adv = pgd_attack(model, x, y, criterion, eps, alpha, iters)
        with torch.no_grad():
            pred = model(x_adv).argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)
    return correct / total

In [10]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 100
EPSILON = 8/255
ALPHA = 2/255
ITERS = 10
PATIENCE = 5  

best_val_score = -1.0
epochs_without_improvement = 0

print("Starting Adversarial Training...")

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct_clean = 0
    total_clean = 0
    
    for batch_idx, (batch_images, batch_labels) in enumerate(train_loader):
        batch_images, batch_labels = batch_images.to(device), batch_labels.to(device)
        
        adv_images = pgd_attack(model, batch_images, batch_labels, criterion, EPSILON, ALPHA, ITERS)
            
    
        train_images = torch.cat((batch_images, adv_images), dim=0)
        train_labels = torch.cat((batch_labels, batch_labels), dim=0)
        
        optimizer.zero_grad()
        outputs = model(train_images)
        loss = criterion(outputs, train_labels)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
        clean_outputs = outputs[:batch_images.size(0)]
        _, predicted = torch.max(clean_outputs.data, 1)
        total_clean += batch_labels.size(0)
        correct_clean += (predicted == batch_labels).sum().item()
            
    epoch_loss = running_loss / len(train_loader)
    train_clean_acc = 100 * correct_clean / total_clean
    

    val_clean_acc = evaluate_clean(model, val_loader)
    val_pgd_acc = evaluate_pgd(model, val_loader, EPSILON, ALPHA, ITERS)
    

    val_score = 0.5 * val_clean_acc + 0.5 * val_pgd_acc
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] | Loss: {epoch_loss:.4f} | Train Clean: {train_clean_acc:.2f}% | Val Clean: {val_clean_acc:.4f} | Val PGD: {val_pgd_acc:.4f} | Val Score: {val_score:.4f}")
    
    #Early stop
    if val_score > best_val_score:
        best_val_score = val_score
        epochs_without_improvement = 0
        torch.save(model.state_dict(), "model_best.pt")
        print("  -> Best model saved!")
    else:
        epochs_without_improvement += 1
        print(f"  -> No improvement for {epochs_without_improvement} epoch(s).")
        
    if epochs_without_improvement >= PATIENCE:
        print(f"\nEarly stopping triggered after {epoch+1} epochs!")
        break

Starting Adversarial Training...
Epoch [1/100] | Loss: 1.0417 | Train Clean: 81.14% | Val Clean: 0.7810 | Val PGD: 0.3680 | Val Score: 0.5745
  -> Best model saved!
Epoch [2/100] | Loss: 1.0233 | Train Clean: 81.74% | Val Clean: 0.7820 | Val PGD: 0.3590 | Val Score: 0.5705
  -> No improvement for 1 epoch(s).
Epoch [3/100] | Loss: 1.0062 | Train Clean: 82.05% | Val Clean: 0.7600 | Val PGD: 0.3420 | Val Score: 0.5510
  -> No improvement for 2 epoch(s).
Epoch [4/100] | Loss: 0.9959 | Train Clean: 82.54% | Val Clean: 0.7920 | Val PGD: 0.3650 | Val Score: 0.5785
  -> Best model saved!
Epoch [5/100] | Loss: 0.9751 | Train Clean: 83.31% | Val Clean: 0.7930 | Val PGD: 0.3570 | Val Score: 0.5750
  -> No improvement for 1 epoch(s).
Epoch [6/100] | Loss: 0.9596 | Train Clean: 83.77% | Val Clean: 0.7660 | Val PGD: 0.3380 | Val Score: 0.5520
  -> No improvement for 2 epoch(s).
Epoch [7/100] | Loss: 1.3517 | Train Clean: 71.31% | Val Clean: 0.7430 | Val PGD: 0.3090 | Val Score: 0.5260
  -> No improv

In [9]:
print(f"\nTraining complete. Best validation score: {best_val_score:.4f}")



Training complete. Best validation score: 0.5765
